In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Cluster Heartbeat - Dashboard Demo\n",
    "\n",
    "## Demonstrating the Cluster Heartbeat Dashboard\n",
    "\n",
    "This notebook shows how to generate and visualize the dashboard data for the Cluster Heartbeat system."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "import os\n",
    "from pathlib import Path\n",
    "\n",
    "sys.path.insert(0, str(Path.cwd().parent))\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from datetime import datetime\n",
    "import json\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.config import load_config\n",
    "from src.core.service import ClusterHeartbeatService\n",
    "from src.data.ingestion import DataIngestion"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load configuration\n",
    "config = load_config('../config/config.yaml')\n",
    "\n",
    "# Initialize service\n",
    "service = ClusterHeartbeatService(config)\n",
    "service.start()\n",
    "\n",
    "print(\"Service started successfully!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load and process data\n",
    "ingestion = DataIngestion(config)\n",
    "df = ingestion.load_data(source='synthetic')\n",
    "\n",
    "# Process data\n",
    "results = service.process_metrics(df.to_dict('records'))\n",
    "\n",
    "print(f\"Processed {len(df)} data points\")\n",
    "print(f\"Results timestamp: {results['timestamp']}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Get dashboard data\n",
    "dashboard = service.get_dashboard_data()\n",
    "\n",
    "print(\"=\"*60)\n",
    "print(\"DASHBOARD SUMMARY\")\n",
    "print(\"=\"*60)\n",
    "print(f\"\\n📊 Cluster Summary:\")\n",
    "print(json.dumps(dashboard['cluster_summary'], indent=2))\n",
    "\n",
    "print(f\"\\n💚 Health Scores:\")\n",
    "print(json.dumps(dashboard['health_scores'], indent=2))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize cluster health\n",
    "fig, axes = plt.subplots(2, 2, figsize=(14, 10))\n",
    "\n",
    "# 1. Health status gauge\n",
    "health_score = dashboard['health_scores'].get('score', 0)\n",
    "status = dashboard['health_scores'].get('status', 'unknown')\n",
    "\n",
    "colors = {'healthy': 'green', 'warning': 'orange', 'degraded': 'red', 'critical': 'darkred'}\n",
    "color = colors.get(status, 'gray')\n",
    "\n",
    "# Gauge chart\n",
    "ax = axes[0, 0]\n",
    "gauge_data = [health_score, 100 - health_score]\n",
    "wedges, texts, autotexts = ax.pie(gauge_data, \n",
    "                                  colors=[color, 'lightgray'],\n",
    "                                  autopct=lambda pct: f'{int(pct)}%' if pct < 50 else '',\n",
    "                                  startangle=90,\n",
    "                                  wedgeprops={'width': 0.3, 'edgecolor': 'white'})\n",
    "ax.text(0, 0, f'{health_score:.1f}%\\n{status.upper()}', \n",
    "        ha='center', va='center', fontsize=14, fontweight='bold')\n",
    "ax.set_title('Cluster Health Score', fontsize=12)\n",
    "\n",
    "# 2. Node health scores\n",
    "ax = axes[0, 1]\n",
    "node_scores = dashboard['health_scores'].get('node_scores', {})\n",
    "if node_scores:\n",
    "    nodes = list(node_scores.keys())\n",
    "    scores = list(node_scores.values())\n",
    "    colors = ['green' if s > 70 else 'orange' if s > 50 else 'red' for s in scores]\n",
    "    ax.barh(nodes, scores, color=colors, edgecolor='black')\n",
    "    ax.set_xlabel('Health Score')\n",
    "    ax.set_title('Node Health Scores', fontsize=12)\n",
    "else:\n",
    "    ax.text(0.5, 0.5, 'No node data available', ha='center', va='center')\n",
    "    ax.set_title('Node Health Scores', fontsize=12)\n",
    "\n",
    "# 3. Anomaly summary\n",
    "ax = axes[1, 0]\n",
    "anomalies = dashboard.get('anomalies', {})\n",
    "anomaly_count = anomalies.get('count', 0)\n",
    "total = dashboard['cluster_summary'].get('gpus', 100)\n",
    "normal_count = total - anomaly_count\n",
    "\n",
    "ax.pie([normal_count, anomaly_count], \n",
    "       labels=['Normal', 'Anomaly'],\n",
    "       colors=['lightgreen', 'red'],\n",
    "       autopct='%1.1f%%',\n",
    "       startangle=90,\n",
    "       wedgeprops={'edgecolor': 'white'})\n",
    "ax.set_title(f'Anomaly Detection ({anomaly_count} anomalies)', fontsize=12)\n",
    "\n",
    "# 4. Cost savings\n",
    "ax = axes[1, 1]\n",
    "cost = dashboard.get('cost_savings', {})\n",
    "idle_gpus = cost.get('idle_gpus', 0)\n",
    "savings = cost.get('potential_savings', 0)\n",
    "total_wasted = cost.get('total_wasted', 0)\n",
    "\n",
    "if idle_gpus > 0:\n",
    "    ax.bar(['Idle GPUs', 'Cost Wasted', 'Potential Savings'], \n",
    "           [idle_gpus, total_wasted, savings],\n",
    "           color=['orange', 'red', 'green'],\n",
    "           edgecolor='black')\n",
    "    ax.set_ylabel('GPUs / USD')\n",
    "    ax.set_title(f'Cost Optimization ({idle_gpus} idle GPUs)', fontsize=12)\n",
    "    \n",
    "    # Add labels\n",
    "    for i, (label, value) in enumerate(zip(['Idle GPUs', 'Cost Wasted', 'Potential Savings'], \n",
    "                                          [idle_gpus, total_wasted, savings])):\n",
    "        ax.text(i, value + 0.1, f'{value:.1f}', ha='center', va='bottom')\n",
    "else:\n",
    "    ax.text(0.5, 0.5, 'No idle GPUs detected! ✅', ha='center', va='center', fontsize=12)\n",
    "    ax.set_title('Cost Optimization', fontsize=12)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Time series visualization\n",
    "timeseries = dashboard.get('timeseries', {})\n",
    "\n",
    "if timeseries and timeseries.get('timestamps'):\n",
    "    fig, axes = plt.subplots(3, 1, figsize=(14, 10))\n",
    "    \n",
    "    timestamps = timeseries['timestamps']\n",
    "    metrics = timeseries.get('metrics', {})\n",
    "    \n",
    "    # GPU Utilization\n",
    "    if 'gpu_utilization' in metrics:\n",
    "        axes[0].plot(timestamps[-100:], metrics['gpu_utilization'][-100:], \n",
    "                     linewidth=2, color='blue')\n",
    "        axes[0].set_ylabel('GPU Utilization')\n",
    "        axes[0].set_title('GPU Utilization Over Time', fontsize=12)\n",
    "        axes[0].grid(True, alpha=0.3)\n",
    "        axes[0].axhline(y=0.8, color='orange', linestyle='--', label='High threshold (80%)')\n",
    "        axes[0].axhline(y=0.2, color='red', linestyle='--', label='Low threshold (20%)')\n",
    "        axes[0].legend()\n",
    "    \n",
    "    # Memory Utilization\n",
    "    if 'memory_utilization' in metrics:\n",
    "        axes[1].plot(timestamps[-100:], metrics['memory_utilization'][-100:], \n",
    "                     linewidth=2, color='green')\n",
    "        axes[1].set_ylabel('Memory Utilization')\n",
    "        axes[1].set_title('Memory Utilization Over Time', fontsize=12)\n",
    "        axes[1].grid(True, alpha=0.3)\n",
    "        axes[1].axhline(y=0.8, color='orange', linestyle='--', label='High threshold (80%)')\n",
    "        axes[1].legend()\n",
    "    \n",
    "    # Temperature\n",
    "    if 'temperature' in metrics:\n",
    "        axes[2].plot(timestamps[-100:], metrics['temperature'][-100:], \n",
    "                     linewidth=2, color='red')\n",
    "        axes[2].set_ylabel('Temperature (°C)')\n",
    "        axes[2].set_xlabel('Time Steps')\n",
    "        axes[2].set_title('GPU Temperature Over Time', fontsize=12)\n",
    "        axes[2].grid(True, alpha=0.3)\n",
    "        axes[2].axhline(y=80, color='orange', linestyle='--', label='Warning (80°C)')\n",
    "        axes[2].axhline(y=90, color='red', linestyle='--', label='Critical (90°C)')\n",
    "        axes[2].legend()\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Get recommendations\n",
    "recommendations = service.get_recommendations()\n",
    "\n",
    "print(\"=\"*60)\n",
    "print(\"RECOMMENDATIONS\")\n",
    "print(\"=\"*60)\n",
    "for rec in recommendations['recommendations']:\n",
    "    print(f\"• {rec}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Export dashboard data as JSON\n",
    "dashboard_json = json.dumps(dashboard, indent=2, default=str)\n",
    "\n",
    "# Save to file\n",
    "with open('../dashboard_output.json', 'w') as f:\n",
    "    f.write(dashboard_json)\n",
    "\n",
    "print(\"✅ Dashboard data saved to dashboard_output.json\")\n",
    "print(f\"File size: {len(dashboard_json):,} bytes\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Service statistics\n",
    "stats = service.get_service_stats()\n",
    "\n",
    "print(\"=\"*60)\n",
    "print(\"SERVICE STATISTICS\")\n",
    "print(\"=\"*60)\n",
    "print(f\"\\n⏱️  Uptime: {stats['uptime_seconds']:.0f} seconds\")\n",
    "print(f\"📊 Total jobs: {stats['total_jobs']}\")\n",
    "print(f\"✅ Processed jobs: {stats['processed_jobs']}\")\n",
    "print(f\"❌ Failed jobs: {stats['failed_jobs']}\")\n",
    "print(f\"⚡ Avg processing time: {stats['avg_processing_time']:.3f}s\")\n",
    "print(f\"💾 Cache size: {stats['cache_size']}\")\n",
    "print(f\"📋 Queue size: {stats['queue_size']}\")\n",
    "print(f\"💚 Health status: {stats['health_status']}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Clean up\n",
    "service.stop()\n",
    "print(\"\\n✅ Service stopped successfully!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Summary\n",
    "print(\"=\"*60)\n",
    "print(\"DASHBOARD DEMO SUMMARY\")\n",
    "print(\"=\"*60)\n",
    "print(f\"\\n✅ Dashboard data generated successfully\")\n",
    "print(f\"✅ Service processed {stats['processed_jobs']} jobs\")\n",
    "print(f\"✅ Recommendations: {len(recommendations['recommendations'])}\")\n",
    "print(f\"✅ Health status: {dashboard['cluster_summary'].get('health_status', 'unknown')}\")\n",
    "print(f\"✅ Dashboard JSON saved\")\n",
    "print(\"\\n📊 Dashboard is ready for Grafana or React integration!\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}